# KAN-39 — Demand Pattern Analysis (Phase 6)

**Phase 6 — Plan.md**: Demand Pattern Metrics & Syntetos-Boylan Classification

## Mục tiêu
- Tính **ADI** (Average Demand Interval) — khoảng cách trung bình giữa các lần có đơn hàng
- Tính **CV²** (Squared Coefficient of Variation) — độ biến thiên của demand dương
- Tính **Zero Rate** — tỷ lệ ngày không phát sinh đơn hàng
- Phân loại theo Syntetos-Boylan (2005): **Smooth / Erratic / Intermittent / Lumpy**

> ⚠️ **QUAN TRỌNG**: Phân loại demand pattern MÔ TẢ HÌNH DẠNG demand, KHÔNG phải ground truth slow-moving.
> `Lumpy ≠ Slow-moving`. ADI/CV² là **mô tả thống kê**, không phải **nhãn kinh doanh**.

## Syntetos-Boylan Classification
| Pattern | ADI | CV² |
|---|---|---|
| Smooth | ≤ 1.32 | ≤ 0.49 |
| Erratic | ≤ 1.32 | > 0.49 |
| Intermittent | > 1.32 | ≤ 0.49 |
| Lumpy | > 1.32 | > 0.49 |

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd()).parent if os.getcwd().endswith('notebooks') else Path(os.getcwd())
sys.path.append(str(PROJECT_ROOT / 'scripts'))
from demand_pattern import compute_demand_patterns

print(f'Project Root: {PROJECT_ROOT}')

Project Root: /home/nguyen-dai-tung/supplyforge-ai


In [2]:
# Load Feature Store (selected columns only for efficiency)
df_daily = pd.read_parquet(
    PROJECT_ROOT / 'data' / 'processed' / 'feature_dataset.parquet',
    columns=['date', 'itemId', 'quantity', 'brand', 'categoryId', 'price', 'cost', 'cost_imputed']
)
print(f'Loaded: {len(df_daily):,} rows | {df_daily["itemId"].nunique()} SKUs')
print(f'Date range: {df_daily["date"].min().date()} -> {df_daily["date"].max().date()}')

Loaded: 1,570,740 rows | 940 SKUs
Date range: 2021-05-20 -> 2025-12-15


In [3]:
# Compute Demand Pattern Metrics per SKU
print('Computing per-SKU ADI, CV², Zero Rate...')
df_patterns = compute_demand_patterns(df_daily)

print('\nDemand Pattern Distribution:')
from IPython.display import display
display(df_patterns['demand_pattern'].value_counts().rename('SKU Count').to_frame())

Computing per-SKU ADI, CV², Zero Rate...



Demand Pattern Distribution:


,SKU Count
demand_pattern,
Intermittent,498
Lumpy,442


In [4]:
# Summary statistics per pattern
summary = df_patterns.groupby('demand_pattern').agg(
    n_skus=('itemId', 'count'),
    mean_adi=('adi', 'mean'),
    mean_cv2=('cv2', 'mean'),
    mean_zero_rate=('zero_rate', 'mean'),
    mean_sales_velocity=('sales_velocity', 'mean')
).round(3)
summary['pct_skus'] = (summary['n_skus'] / summary['n_skus'].sum() * 100).round(1)
print('Syntetos-Boylan Pattern Summary:')
display(summary[['n_skus', 'pct_skus', 'mean_adi', 'mean_cv2', 'mean_zero_rate', 'mean_sales_velocity']])

Syntetos-Boylan Pattern Summary:


,n_skus,pct_skus,mean_adi,mean_cv2,mean_zero_rate,mean_sales_velocity
demand_pattern,,,,,,
Intermittent,498,53.0,557.461,0.145,0.982,0.028
Lumpy,442,47.0,46.356,4.194,0.900,1.250


In [5]:
# Visualization: ADI vs CV² scatter + Pie chart
colors = {'Smooth': '#2ecc71', 'Erratic': '#f39c12', 'Intermittent': '#3498db', 'Lumpy': '#e74c3c'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=130)

# Scatter plot
for pattern, grp in df_patterns.groupby('demand_pattern'):
    adi_capped = grp['adi'].clip(upper=10)
    axes[0].scatter(adi_capped, grp['cv2'], c=colors.get(pattern, 'gray'),
                    label=pattern, alpha=0.6, s=25, edgecolors='white', linewidths=0.3)

axes[0].axvline(x=1.32, color='gray', linestyle='--', linewidth=1.2, label='ADI = 1.32')
axes[0].axhline(y=0.49, color='gray', linestyle=':', linewidth=1.2, label='CV² = 0.49')
axes[0].set_xlabel('ADI (capped at 10)', fontsize=11)
axes[0].set_ylabel('CV²', fontsize=11)
axes[0].set_title('Syntetos-Boylan Demand Pattern Space\n(940 SKUs)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Pie chart
cnts = df_patterns['demand_pattern'].value_counts()
axes[1].pie(
    cnts.values,
    labels=[f"{p}\n{v} SKUs ({v/len(df_patterns)*100:.1f}%)" for p,v in zip(cnts.index, cnts.values)],
    colors=[colors.get(p,'gray') for p in cnts.index],
    startangle=90, wedgeprops={'edgecolor':'white', 'linewidth':1.5}
)
axes[1].set_title('Demand Pattern Distribution', fontsize=12, fontweight='bold')

plt.suptitle('SupplyForge AI — Phase 6: Demand Pattern Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports' / 'demand_pattern_chart.png', bbox_inches='tight')
plt.show()
print('Chart saved.')

Chart saved.


/tmp/ipykernel_45354/879146113.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Export demand_patterns.parquet
out_path = PROJECT_ROOT / 'data' / 'processed' / 'demand_patterns.parquet'
df_patterns.to_parquet(out_path, index=False, compression='snappy')
print(f'Saved: {out_path.name} ({out_path.stat().st_size/1024:.0f} KB, {len(df_patterns)} SKUs)')

# Also save summary CSV
summary.to_csv(PROJECT_ROOT / 'reports' / 'demand_pattern_summary.csv')
print('Saved: demand_pattern_summary.csv')

Saved: demand_patterns.parquet (44 KB, 940 SKUs)
Saved: demand_pattern_summary.csv
